# HGP(58,16,3) CNOT Logical Error Rates

This notebook benchmarks the logical CNOT experiment for the [[58,16,3]] hypergraph-product code using the same ancilla-mediated homological-measurement construction as in `hgp58_16_3.ipynb`. Visualization is intentionally skipped; only logical-rate data is saved.

In [1]:
import importlib
import numpy as np
import sinter
import stim
from ldpc import SinterBpOsdDecoder

from pathlib import Path

try:
    qec_code_constructions = importlib.import_module("qec.code_constructions")
    HypergraphProductCode = qec_code_constructions.HypergraphProductCode
except ModuleNotFoundError as exc:
    raise ImportError(
        "Missing optional dependency 'qec'. Install it in this environment (for example: pip install qec)."
    ) from exc

from epic import (
    AllocCode,
    FreeCode,
    InitCode,
    PauliChar,
    PauliEigenState,
    PauliString,
    QECCompiler,
    ReadoutCode,
    StimLikeNoiseModel,
)
from epic.modules.qec_gadgets.pauli_product_measurement.homological_measurement import HomologicalMeasurement
from epic.modules.stabilizers_codes.css_code import CSSCode
from epic.modules.stabilizers_codes.rotated_surface_code import RotatedSurfaceCode
from epic.modules.stabilizers_codes.utils.hgp_canonical_basis_util import get_canonical_basis

base_dir = Path("/Users/lenny/Documents/foo/EPIC-QEC/docs/example_notebooks/cnot_benchmark_result")
logical_rates_dir = base_dir / "logical_rates"
logical_rates_dir.mkdir(parents=True, exist_ok=True)

hamming_code = np.array(
    [
        [1, 0, 0, 1, 0, 1, 1],
        [0, 1, 0, 1, 1, 0, 1],
        [0, 0, 1, 0, 1, 1, 1],
    ],
    dtype=np.uint8,
)

example_code = HypergraphProductCode(hamming_code, hamming_code)
hx = example_code.x_stabilizer_matrix.toarray().astype(np.uint8)
hz = example_code.z_stabilizer_matrix.toarray().astype(np.uint8)

canon_basis = get_canonical_basis(hamming_code, hamming_code)
sorted_keys = sorted(canon_basis.keys())
logical_qubits = [
    (canon_basis[k][0].astype(int).tolist(), canon_basis[k][1].astype(int).tolist())
    for k in sorted_keys
]
k_lq = len(logical_qubits)

control_code = CSSCode.from_css_pcm("hgp58_16_3_patch", hx, hz, logical_qubits)
ancilla_code = RotatedSurfaceCode.from_distance(distance=3, code_name="ancilla_d3", system_coordinate=(1, 0))

PRIMITIVES_CONFIG = {
    "epic.core.qec_primitives.interfaces.ApplyGate": "epic.modules.qec_primitives.apply_gates.simple_gate_application.SimpleGateApplication",
    "epic.core.qec_primitives.interfaces.Readout": "epic.modules.qec_primitives.readouts.naive_readout.NaiveReadout",
    "epic.core.qec_primitives.interfaces.ExtractSyndrome": "epic.modules.qec_primitives.syndrome_extraction.zxcoloring_extraction.ZXColoringExtraction",
}


def compile_hgp_cnot_experiment():
    ctrl_lq_names = [f"p1_lq_{i}" for i in range(k_lq)]
    anc_lq_names = [f"anc_lq_{i}" for i in range(k_lq)]

    program = [
        AllocCode(target_code=control_code, code_varname="patch1", logical_qubits_varnames=ctrl_lq_names),
        AllocCode(target_code=ancilla_code, code_varname="anc_patch", logical_qubits_varnames=anc_lq_names),
        InitCode(targets=["patch1"], initial_state=PauliEigenState.Z_plus, tag="init_ctrl"),
        InitCode(targets=["anc_patch"], initial_state=PauliEigenState.X_plus, tag="init_anc"),
        HomologicalMeasurement(
            targets=["p1_lq_0", "anc_lq_0"],
            product_to_measure=PauliString(string=(PauliChar.Z, PauliChar.Z)),
            tag="MZZ",
        ),
        HomologicalMeasurement(
            targets=["anc_lq_0", "p1_lq_1"],
            product_to_measure=PauliString(string=(PauliChar.X, PauliChar.X)),
            tag="MXX",
        ),
        ReadoutCode(targets=["patch1"], tag="ro_ctrl"),
        ReadoutCode(targets=["anc_patch"], tag="ro_anc"),
        FreeCode(code_varname="patch1"),
        FreeCode(code_varname="anc_patch"),
    ]

    compiler = QECCompiler(
        config={
            "objective_distance": 3,
            "primitives": PRIMITIVES_CONFIG,
        }
    )
    compiled_program = compiler.compile(program, show_progress=False)

    # Follow the same feedforward convention as in docs/example_notebooks/hgp58_16_3.ipynb,
    # but discover exact observable tags emitted by the compiler.
    emitted_tags = [ob.tag for ob in compiled_program.observables]
    readout_tags = [tag for tag in emitted_tags if tag.startswith("readout_")]

    def logical_index(tag: str) -> int:
        marker = "_lq_"
        if marker not in tag:
            raise ValueError(f"Could not parse logical index from tag: {tag}")
        suffix = tag.rsplit(marker, 1)[1]
        if not suffix.isdigit():
            raise ValueError(f"Could not parse logical index from tag: {tag}")
        return int(suffix)

    ancilla_readouts = [tag for tag in readout_tags if "anc" in tag.lower()]
    data_readouts = [tag for tag in readout_tags if "anc" not in tag.lower()]

    control_candidates = [tag for tag in data_readouts if logical_index(tag) == 0]
    target_candidates = [tag for tag in data_readouts if logical_index(tag) == 1]
    ancilla_candidates = [tag for tag in ancilla_readouts if logical_index(tag) == 0]
    mzz_candidates = [tag for tag in emitted_tags if "MZZ" in tag]

    if len(control_candidates) != 1 or len(target_candidates) != 1:
        raise ValueError(
            "Could not uniquely identify control/target readout tags. "
            f"Control candidates={control_candidates}, target candidates={target_candidates}"
        )
    if len(ancilla_candidates) != 1:
        raise ValueError(f"Could not uniquely identify ancilla readout tag: {ancilla_candidates}")
    if len(mzz_candidates) != 1:
        raise ValueError(f"Could not uniquely identify MZZ observable tag: {mzz_candidates}")

    control_tag = control_candidates[0]
    target_tag = target_candidates[0]
    ancilla_tag = ancilla_candidates[0]
    mzz_tag = mzz_candidates[0]

    unused_data_readouts = sorted(
        [tag for tag in data_readouts if logical_index(tag) >= 2],
        key=logical_index,
    )

    stim_observables = [
        [control_tag],
        [mzz_tag, ancilla_tag, target_tag],
    ]
    for tag in unused_data_readouts:
        stim_observables.append([tag])

    return compiled_program, stim_observables


noise_values = np.logspace(-5, -1, 9)
shots = 100

compiled_program, stim_observables = compile_hgp_cnot_experiment()

tasks = []
for physical_noise in noise_values:
    noise_model = StimLikeNoiseModel.from_stim_like_probabilities(
        after_clifford_depolarization=float(physical_noise),
        after_reset_flip_probability=float(physical_noise),
        before_measure_flip_probability=float(physical_noise),
        before_round_data_depolarization=float(physical_noise),
    )
    stim_program = compiled_program.to_stim_program(stim_observables, noise_model=noise_model)
    tasks.append(
        sinter.Task(
            circuit=stim.Circuit(stim_program),
            json_metadata={"d": 3, "p": float(physical_noise), "code": "hgp58_16_3"},
        )
    )

print("Starting BPOSD benchmark run")
print(f"  shots={shots}, noise_points={len(noise_values)}, decoder=bposd")

def log_progress(progress: sinter.Progress) -> None:
    if progress.status_message:
        print(f"[bposd] {progress.status_message}")

collected_stats = sinter.collect(
    num_workers=4,
    tasks=tasks,
    decoders=["bposd"],
    custom_decoders={
        "bposd": SinterBpOsdDecoder(
            max_iter=0,
            bp_method="ms",
            ms_scaling_factor=0.625,
            osd_method="osd0",
            osd_order=0,
        )
    },
    max_shots=shots,
    progress_callback=log_progress,
    print_progress=True,
)
stats_by_noise = {float(stats.json_metadata["p"]): stats for stats in collected_stats}
logical_error_rates = np.array(
    [
        stats_by_noise[float(physical_noise)].errors / stats_by_noise[float(physical_noise)].shots
        for physical_noise in noise_values
    ]
)

result_title = "EPIC HGP(58,16,3) CNOT logical error rate"
result_description = (
    "Logical failure probability for ancilla-mediated CNOT on HGP(58,16,3), "
    "using HM-based ZZ/XX product measurements with feedforward observable definition. "
    "A shot is counted as a logical fail when any tracked observable is 1. "
    "Noise uses EPIC StimLikeNoiseModel with matched p for reset flip, Clifford depolarization, "
    "pre-measurement flip, and round-data depolarization channels."
)
results_path = logical_rates_dir / "hgp58_16_3_cnot_rates.npz"
np.savez_compressed(
    results_path,
    distance=np.array([3], dtype=np.int64),
    noise_values=noise_values,
    logical_error_rates=logical_error_rates,
    shots=np.array([shots], dtype=np.int64),
    code=np.array(["hgp58_16_3"]),
    title=np.array([result_title]),
    description=np.array([result_description]),
)

print(f"Saved HGP(58,16,3) CNOT rates to: {results_path.name}")
for physical_noise, logical_error_rate in zip(noise_values, logical_error_rates):
    print(f"  p={physical_noise:.2e} -> P_L={logical_error_rate:.6f}")


DEBUG:matplotlib:CACHEDIR=/Users/lenny/.matplotlib
DEBUG:matplotlib.font_manager:Using fontManager instance from /Users/lenny/.matplotlib/fontlist-v390.json
Starting 4 workers...


Starting BPOSD benchmark run
  shots=100, noise_points=9, decoder=bposd
[bposd] Starting 4 workers...


9 tasks left:
  workers decoder eta shots_left errors_seen json_metadata                               
        1   bposd   ?        100           0 d=3,p=1e-05,code=hgp58_16_3                 
        1   bposd  1m         99           0 d=3,p=3.1622776601683795e-05,code=hgp58_16_3
        1   bposd   ?        100           0 d=3,p=0.0001,code=hgp58_16_3                
        1   bposd   ?        100           0 d=3,p=0.00031622776601683794,code=hgp58_16_3
        0   bposd ?·∞        100           0 d=3,p=0.001,code=hgp58_16_3                 
        0   bposd ?·∞        100           0 d=3,p=0.0031622776601683794,code=hgp58_16_3 
        0   bposd ?·∞        100           0 d=3,p=0.01,code=hgp58_16_3                  
        0   bposd ?·∞        100           0 d=3,p=0.03162277660168379,code=hgp58_16_3   
        0   bposd ?·∞        100           0 d=3,p=0.1,code=hgp58_16_3                   
9 tasks left:
  workers decoder eta shots_left errors_seen json_metadata              

[bposd] 9 tasks left:
  workers decoder eta shots_left errors_seen json_metadata                               
        1   bposd   ?        100           0 d=3,p=1e-05,code=hgp58_16_3                 
        1   bposd  1m         99           0 d=3,p=3.1622776601683795e-05,code=hgp58_16_3
        1   bposd   ?        100           0 d=3,p=0.0001,code=hgp58_16_3                
        1   bposd   ?        100           0 d=3,p=0.00031622776601683794,code=hgp58_16_3
        0   bposd ?·∞        100           0 d=3,p=0.001,code=hgp58_16_3                 
        0   bposd ?·∞        100           0 d=3,p=0.0031622776601683794,code=hgp58_16_3 
        0   bposd ?·∞        100           0 d=3,p=0.01,code=hgp58_16_3                  
        0   bposd ?·∞        100           0 d=3,p=0.03162277660168379,code=hgp58_16_3   
        0   bposd ?·∞        100           0 d=3,p=0.1,code=hgp58_16_3                   
[bposd] 9 tasks left:
  workers decoder eta shots_left errors_seen json_metada

9 tasks left:
  workers decoder eta shots_left errors_seen json_metadata                               
        1   bposd  1m         99           0 d=3,p=1e-05,code=hgp58_16_3                 
        1   bposd  1m         99           0 d=3,p=3.1622776601683795e-05,code=hgp58_16_3
        1   bposd  1m         98           0 d=3,p=0.0001,code=hgp58_16_3                
        1   bposd   ?        100           0 d=3,p=0.00031622776601683794,code=hgp58_16_3
        0   bposd ?·∞        100           0 d=3,p=0.001,code=hgp58_16_3                 
        0   bposd ?·∞        100           0 d=3,p=0.0031622776601683794,code=hgp58_16_3 
        0   bposd ?·∞        100           0 d=3,p=0.01,code=hgp58_16_3                  
        0   bposd ?·∞        100           0 d=3,p=0.03162277660168379,code=hgp58_16_3   
        0   bposd ?·∞        100           0 d=3,p=0.1,code=hgp58_16_3                   
9 tasks left:
  workers decoder eta shots_left errors_seen json_metadata              

...

6 tasks left:
  workers decoder eta shots_left errors_seen json_metadata                               
        1   bposd <1m          2           1 d=3,p=0.00031622776601683794,code=hgp58_16_3
        1   bposd 48m         60           0 d=3,p=0.001,code=hgp58_16_3                 
        1   bposd 62m         66          15 d=3,p=0.0031622776601683794,code=hgp58_16_3 
        1   bposd 71m         76          24 d=3,p=0.01,code=hgp58_16_3                  
        0   bposd ?·∞        100           0 d=3,p=0.03162277660168379,code=hgp58_16_3   
        0   bposd ?·∞        100           0 d=3,p=0.1,code=hgp58_16_3                   


[bposd] 6 tasks left:
  workers decoder eta shots_left errors_seen json_metadata                               
        1   bposd <1m          2           1 d=3,p=0.00031622776601683794,code=hgp58_16_3
        1   bposd 48m         60           0 d=3,p=0.001,code=hgp58_16_3                 
        1   bposd 62m         66          15 d=3,p=0.0031622776601683794,code=hgp58_16_3 
        1   bposd 71m         76          24 d=3,p=0.01,code=hgp58_16_3                  
        0   bposd ?·∞        100           0 d=3,p=0.03162277660168379,code=hgp58_16_3   
        0   bposd ?·∞        100           0 d=3,p=0.1,code=hgp58_16_3                   


KeyboardInterrupt


[bposd] KeyboardInterrupt


KeyboardInterrupt: 